[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/kepler/kepler.ipynb)

# Through periapsis with a position spinor

A highly eccentric orbit crosses periapsis so quickly that a fixed physical-time step can miss the turn. Carry the position as a scaled spinor and let physical time advance by the radius: the inverse-square motion becomes a smooth oscillator. We integrate both descriptions and measure their errors against the exact bound orbit.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.mechanics.kepler import render
from examples.mechanics.kepler.core import State, BoundOrbit

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Spinor = ga.gatype.even()

In [ ]:
gravity = 1.0
semimajor_axis = 1.0
eccentricities = np.array([0.6, 0.9, 0.97])
labels = ("Eccentricity 0.6", "Eccentricity 0.9", "Eccentricity 0.97")
featured_case = 2
lift_samples = 64

# Start at apoapsis. The spinor's squared size is distance; its sandwich gives position.
distance = semimajor_axis * (1 + eccentricities)
speed = np.sqrt(gravity / semimajor_axis * (1 - eccentricities) / (1 + eccentricities))
spinor = mv.rotor() * np.sqrt(distance)                                  # [cases] Spinor
velocity = mv.y * speed                                                 # [cases] Vector
energy = velocity.scalar_norm_squared() / 2 - gravity / spinor.scalar_norm_squared()
# Lift the physical velocity without introducing rotation about the radial direction.
rate = velocity * spinor * mv.x / 2                                      # [cases] Spinor
orbit = BoundOrbit(spinor, rate, energy)
frequency = (-energy / 2).square_root()                                  # [cases] Scalar

# Equal regularized steps resolve periapsis. One physical orbit is half a spinor cycle.
phase = np.linspace(0, np.pi, lift_samples + 1)[:, None]
spinors = spinor * np.cos(phase) + rate / frequency * np.sin(phase)        # [samples + 1, cases] Spinor
positions = spinors >> mv.x                                             # [samples + 1, cases] Vector
render.draw_lift(positions[:, featured_case], spinors[:, featured_case]);

The spinor turns and scales the fixed reference direction `mv.x`; its extra freedom is rotation about that direction, which leaves position unchanged. Negative orbital energy makes its regularized acceleration proportional to the spinor itself, with a negative coefficient. The physical clock advances by the squared spinor size, so an equal regularized step covers much less physical time near periapsis. The points above are equally spaced in that clock.

In [ ]:
steps = 128
reference_samples = 1024
period = 2 * np.pi * np.sqrt(semimajor_axis ** 3 / gravity)

def physical_state(spinor: Spinor, rate: Spinor, time: Scalar) -> State:
    """Read a position spinor and its regularized derivative in physical time."""
    radius = spinor.scalar_norm_squared()
    position = spinor >> mv.x
    # The vector part differentiates the sandwich. The trivector is the unused gauge rate.
    velocity = 2 * (rate * mv.x * spinor.reverse()).select[1] / radius
    return State(time, position, velocity)


def exact_orbit(parameter: Scalar) -> State:
    """Exact oscillator evolution and its integrated physical clock, for bound motion."""
    frequency = (-energy / 2).square_root()
    phase = frequency * parameter
    cosine, sine = phase.cos(), phase.sin()
    current_spinor = spinor * cosine + rate * (sine / frequency)
    current_rate = rate * cosine - spinor * (frequency * sine)
    # Physical time advances by the radius: the clock slows smoothly near the focus.
    initial_radius = spinor.scalar_norm_squared()
    rate_radius = rate.scalar_norm_squared() / frequency.squared()
    overlap = spinor.scalar_product(rate.reverse()) / frequency
    time = ((initial_radius + rate_radius) * parameter / 2
            + (initial_radius - rate_radius) * (2 * phase).sin() / (4 * frequency)
            + overlap * sine.squared() / frequency)
    return physical_state(current_spinor, current_rate, time)


def physical_verlet(initial: State, gravity: float, step: float, steps: int) -> Iterator[State]:
    """Kick-drift-kick integration with equal steps in physical time."""
    position, velocity, time = initial.position, initial.velocity, initial.time
    acceleration = -gravity * position / position.norm() ** 3
    yield initial
    for _ in range(steps):
        half_velocity = velocity + acceleration * (step / 2)
        position = position + half_velocity * step
        acceleration = -gravity * position / position.norm() ** 3
        velocity = half_velocity + acceleration * (step / 2)
        time = time + step
        yield State(time, position, velocity)


def regularized_verlet(orbit: BoundOrbit, step: Scalar, steps: int) -> Iterator[State]:
    """Kick-drift-kick integration of the spinor oscillator, with its physical clock."""
    spinor, rate = orbit.spinor, orbit.rate
    time = orbit.energy * 0
    yield physical_state(spinor, rate, time)
    for _ in range(steps):
        half_rate = rate + orbit.energy * spinor * (step / 4)
        following = spinor + half_rate * step
        # Integrate the squared spinor along the linear drift, including the cross term.
        time = time + step / 3 * (spinor.scalar_norm_squared()
                                  + spinor.scalar_product(following.reverse())
                                  + following.scalar_norm_squared())
        rate = half_rate + orbit.energy * following * (step / 4)
        spinor = following
        yield physical_state(spinor, rate, time)


parameter = mv.scalar(np.linspace(0, np.pi, reference_samples + 1)[:, None, None]) / frequency
reference = exact_orbit(parameter)
initial = physical_state(spinor, rate, energy * 0)
physical = State.collect(physical_verlet(initial, gravity, period / steps, steps))
regularized = State.collect(regularized_verlet(orbit, np.pi / frequency / steps, steps))
render.draw_comparison(reference, physical, regularized, labels);

Both numerical paths use kick–drift–kick Verlet and the same step count. The spinor oscillator has a constant coefficient instead of an inverse-cube acceleration factor; its clock concentrates physical-time resolution near the close passage. The lower row enlarges periapsis. This is a fixed-step comparison: an adaptive physical-time solver could also resolve the encounter. Equal step counts do not imply equal wall-clock cost, and regularization does not make the numerical solution exact.

In [ ]:
counts = np.array([64, 128, 256, 512])
clock_iterations = 44


def position_error(trajectory: State) -> Scalar:
    # Invert the exact clock by bisection, comparing at each method's physical sample times.
    exact = orbit.at_time(trajectory.time, clock_iterations)
    squared = (trajectory.position - exact.position).scalar_norm_squared()
    intervals = trajectory.time[1:] - trajectory.time[:-1]
    # Weight by elapsed physical time so clustered periapsis samples do not dominate the norm.
    integral = ((squared[1:] + squared[:-1]) * intervals / 2).sum(axis=0)
    return (integral / (trajectory.time[-1] - trajectory.time[0])).square_root() / semimajor_axis


physical_errors = stack([position_error(State.collect(physical_verlet(
    initial, gravity, period / count, count))) for count in counts])
regularized_errors = stack([position_error(State.collect(regularized_verlet(
    orbit, np.pi / frequency / count, count))) for count in counts])
render.draw_errors(counts, physical_errors, regularized_errors, labels);

The error is a physical-time-weighted RMS position difference, measured against the analytic orbit at the numerical trajectory’s own times. Both schemes converge quadratically when resolved; the eccentric cases expose how much resolution a fixed physical clock needs. The oscillator method preserves its angular-momentum invariants, but its physical Kepler energy still has integration error.

In [ ]:
animation_frames = 120
duration_ms = 60
revolutions = 2

phase = np.linspace(0, revolutions * np.pi, animation_frames + 1)[:, None]
spinors = spinor * np.cos(phase) + rate / frequency * np.sin(phase)
positions = spinors >> mv.x
render.inline(render.animate_lift(positions[:, featured_case], spinors[:, featured_case]), duration_ms)

The animation advances uniformly in the regularized clock, not physical time. The orbit closes twice while the spinor closes once: opposite spinors carry the reference direction to the same position. The construction applies in three spatial dimensions; the displayed orbit lies in the xy plane. This example treats bound, unperturbed Newtonian motion away from exact collision; physical velocity remains singular at collision even though the spinor equation is regular. See [Hestenes and Lounesto, *Geometry of Spinor Regularization*](https://davidhestenes.net/misc/Hestenes_1983_-_Geometry_of_Spinor_Regularization.pdf).